# Homework: Vector Search

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [2]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
filenames = ["embedder.py", "ingest.py"]
dp = None
for filename in filenames:
    for directory in [current, *current.parents]:
        if (directory / "embed" / filename).exists():
            dp = "embed"
            sys.path.insert(0, str(directory))
            break
        if (directory / "src" / filename).exists():
            dp = "src"
            sys.path.insert(0, str(directory))
            break
    else:
        raise FileNotFoundError(f"Cannot find {dp}/{filename}")

## Q1. Embedding a query

In [3]:
from embed.embedder import Embedder

embed = Embedder(
    path="../models/Xenova/all-MiniLM-L6-v2"
)

q = "How does approximate nearest neighbor search work?"

2026-06-30 17:04:06.661595537 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [4]:
v = embed.encode(q)
v[0]

np.float64(-0.02058203437252893)

## Q2. Cosine similarity

In [5]:
to_match = "02-vector-search/lessons/07-sqlitesearch-vector.md"
matched = [
    document
    for document in documents
    if document.get("filename") == to_match
]

In [6]:
matched[0]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [7]:
v_matched = embed.encode(matched[0]['content'])

In [8]:
v_matched.dot(v)

np.float64(0.36107026789538205)

## Q3. Chunking and search by hand

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [10]:
#chunks[0]

In [11]:
X = embed.encode_batch([chunk["content"] for chunk in chunks])

In [12]:
X.shape

(295, 384)

In [13]:
scores = X.dot(v)

In [15]:
import numpy as np
idx = np.argmax(scores)
print(idx, scores[idx])
print(chunks[idx]['filename'])

94 0.6489016436447387
02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch

In [16]:
from minsearch import VectorSearch

vindex = VectorSearch()
vindex.fit(X, chunks)

In [17]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

vector_results = vindex.search(query_vector, num_results=5)

In [18]:
vector_results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

In [19]:
from minsearch import Index

# Create and fit the index
index = Index(
    text_fields=["content"]
)
index.fit(chunks)

# Search
query = "How do I store vectors in PostgreSQL?"

text_results = index.search(query)

In [20]:
text_results[0]['filename']

'02-vector-search/lessons/02-embeddings.md'

## Q6. Hybrid search

In [21]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [22]:
query = "How do I give the model access to tools?"
vector_results = vindex.search(embed.encode(query), num_results=5)
text_results = index.search(query)
results = rrf([vector_results, text_results])

In [23]:
results[0]['filename']

'01-agentic-rag/lessons/13-function-calling.md'